# Zonaprop scrapping

In [ ]:
"""
Zonaprop Scraper - Extrae propiedades de archivos HTML descargados de Zonaprop
Ejecutar en Google Colab con los archivos HTML en base_zonaprop.zip
"""

import zipfile
import re
import pandas as pd
from bs4 import BeautifulSoup

# ── 1. Leer el ZIP ────────────────────────────────────────────────────────────
ruta_zip = "base_zonaprop.zip"

with zipfile.ZipFile(ruta_zip, 'r') as z:
    archivos = z.namelist()

print(f"Total de archivos en el ZIP: {len(archivos)}")


# ── 2. Funciones auxiliares ───────────────────────────────────────────────────

def detectar_tipo(titulo_html: str) -> str:
    """Detecta si el HTML corresponde a Casas o Departamentos según el título."""
    t = titulo_html.lower()
    if "casas" in t:
        return "Casa"
    if "departamentos" in t or "departamento" in t:
        return "Departamento"
    return "Desconocido"


def limpiar_precio(texto: str) -> int | None:
    """Convierte 'USD 290.000' → 290000. Descarta si no tiene USD."""
    if not texto or "USD" not in texto.upper():
        return None
    nums = re.sub(r"[^\d]", "", texto)
    return int(nums) if nums else None


def extraer_feature(spans, keyword: str) -> int | None:
    """Busca 'N keyword' en la lista de spans y devuelve N como int."""
    for s in spans:
        txt = s.get_text(strip=True)
        if keyword in txt:
            num = re.search(r"(\d+(?:[.,]\d+)?)", txt)
            if num:
                return int(float(num.group(1).replace(",", ".")))
    return None


def parsear_html(nombre_archivo: str, contenido_bytes: bytes) -> list[dict]:
    """Parsea un HTML de Zonaprop y devuelve lista de dicts con las propiedades."""
    soup = BeautifulSoup(contenido_bytes, "html.parser")

    # Tipo: del <title> de la página
    title_tag = soup.find("title")
    tipo = detectar_tipo(title_tag.get_text() if title_tag else "")

    resultados = []

    # Cada propiedad está en un div con data-qa="posting PROPERTY" o "posting DEVELOPMENT"
    cards = soup.find_all("div", attrs={"data-qa": re.compile(r"^posting ")})

    for card in cards:
        datos = {"tipo": tipo, "html": nombre_archivo}

        # ── Precio ────────────────────────────────────────────────────────────
        precio_tag = card.find(attrs={"data-qa": "POSTING_CARD_PRICE"})
        if precio_tag:
            datos["valor"] = limpiar_precio(precio_tag.get_text(strip=True))
        else:
            datos["valor"] = None

        # ── Features (m², ambientes, dorm., baños) ───────────────────────────
        features_tag = card.find(attrs={"data-qa": "POSTING_CARD_FEATURES"})
        spans = features_tag.find_all("span") if features_tag else []

        datos["m2"] = extraer_feature(spans, "m²")
        datos["ambientes"] = extraer_feature(spans, "amb.")
        datos["dormitorios"] = extraer_feature(spans, "dorm.")
        datos["baños"] = extraer_feature(spans, "baño")
        datos["cocheras"] = extraer_feature(spans, "coch.")

        # ── Dirección ─────────────────────────────────────────────────────────
        dir_tag = card.find(
            "h4",
            class_=lambda c: c and "location-address" in c and "location-text" not in c
        )
        datos["direccion"] = dir_tag.get_text(strip=True) if dir_tag else None

        # ── Ubicación ─────────────────────────────────────────────────────────
        loc_tag = card.find(attrs={"data-qa": "POSTING_CARD_LOCATION"})
        if loc_tag:
            texto_loc = loc_tag.get_text(strip=True)
            # Extraer barrio: lo que está antes de ", Capital Federal"
            partes = texto_loc.split(",")
            datos["ubicacion"] = partes[0].strip() if partes else texto_loc
        else:
            datos["ubicacion"] = None

        # Solo agregar si tiene al menos precio o dirección
        if datos["valor"] is not None or datos["direccion"] is not None:
            resultados.append(datos)

    return resultados


# ── 3. Procesar todos los archivos ────────────────────────────────────────────

todas_las_propiedades = []

with zipfile.ZipFile(ruta_zip, 'r') as z:
    html_files = [f for f in z.namelist() if f.lower().endswith(".html")]
    print(f"Archivos HTML encontrados: {len(html_files)}")

    for i, nombre in enumerate(html_files, 1):
        try:
            with z.open(nombre) as f:
                contenido = f.read()
            propiedades = parsear_html(nombre, contenido)
            todas_las_propiedades.extend(propiedades)
            if i % 10 == 0:
                print(f"  Procesados {i}/{len(html_files)} archivos... ({len(todas_las_propiedades)} propiedades)")
        except Exception as e:
            print(f"  ⚠ Error en {nombre}: {e}")

print(f"\nTotal propiedades extraídas (con duplicados): {len(todas_las_propiedades)}")


# ── 4. Crear DataFrame y limpiar ─────────────────────────────────────────────

df = pd.DataFrame(todas_las_propiedades, columns=[
    "tipo", "ubicacion", "valor", "m2", "ambientes",
    "dormitorios", "baños", "cocheras", "direccion", "html"
])

# Eliminar duplicados exactos (misma dirección + precio + HTML)
df_unico = df.drop_duplicates(subset=["direccion", "valor", "html"]).reset_index(drop=True)

print(f"Propiedades únicas: {len(df_unico)}")
print(f"\nDistribución por tipo:")
print(df_unico["tipo"].value_counts())
print(f"\nDistribución por ubicación:")
print(df_unico["ubicacion"].value_counts())


# ── 5. Exportar a Excel ───────────────────────────────────────────────────────

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

output_path = "zonaprop_propiedades.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Propiedades"

# Encabezados
columnas = ["tipo", "ubicacion", "valor", "m2", "ambientes", "dormitorios", "baños", "cocheras", "direccion", "html"]
encabezados = ["Tipo", "Ubicación", "Valor (USD)", "m² Totales", "Ambientes", "Dormitorios", "Baños", "Cocheras", "Dirección", "Archivo HTML"]

header_fill = PatternFill("solid", start_color="2E4057")
header_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
center = Alignment(horizontal="center", vertical="center")

thin = Side(style="thin", color="CCCCCC")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for col_idx, (col, encabezado) in enumerate(zip(columnas, encabezados), 1):
    cell = ws.cell(row=1, column=col_idx, value=encabezado)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = center
    cell.border = border

ws.row_dimensions[1].height = 20

# Datos
fill_par = PatternFill("solid", start_color="F5F5F5")
fill_impar = PatternFill("solid", start_color="FFFFFF")
font_data = Font(name="Arial", size=9)

for row_idx, row in df_unico.iterrows():
    fill = fill_par if (row_idx % 2 == 0) else fill_impar
    for col_idx, col in enumerate(columnas, 1):
        val = row[col]
        # Convertir NaN a None para Excel
        if pd.isna(val) if not isinstance(val, str) else False:
            val = None
        cell = ws.cell(row=row_idx + 2, column=col_idx, value=val)
        cell.font = font_data
        cell.fill = fill
        cell.border = border
        if col_idx in (3, 4, 5, 6, 7, 8):
            cell.alignment = Alignment(horizontal="center")

# Anchos de columna
anchos = {1: 14, 2: 20, 3: 14, 4: 12, 5: 12, 6: 14, 7: 10, 8: 12, 9: 35, 10: 60}
for col_idx, ancho in anchos.items():
    ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = ancho

# Hoja de resumen
ws2 = wb.create_sheet("Resumen")
ws2["A1"] = "Resumen de propiedades extraídas"
ws2["A1"].font = Font(bold=True, name="Arial", size=12)

ws2["A3"] = "Total propiedades"
ws2["B3"] = len(df_unico)
ws2["A4"] = "Casas"
ws2["B4"] = int((df_unico["tipo"] == "Casa").sum())
ws2["A5"] = "Departamentos"
ws2["B5"] = int((df_unico["tipo"] == "Departamento").sum())
ws2["A7"] = "Precio promedio (USD)"
ws2["B7"] = f'=AVERAGE(Propiedades!C2:C{len(df_unico)+1})'
ws2["A8"] = "Precio mínimo (USD)"
ws2["B8"] = f'=MIN(Propiedades!C2:C{len(df_unico)+1})'
ws2["A9"] = "Precio máximo (USD)"
ws2["B9"] = f'=MAX(Propiedades!C2:C{len(df_unico)+1})'
ws2["A11"] = "m² promedio"
ws2["B11"] = f'=AVERAGE(Propiedades!D2:D{len(df_unico)+1})'

for r in range(3, 12):
    ws2.cell(row=r, column=1).font = Font(name="Arial", size=10)
    ws2.cell(row=r, column=2).font = Font(bold=True, name="Arial", size=10)

ws2.column_dimensions["A"].width = 25
ws2.column_dimensions["B"].width = 18

wb.save(output_path)
print(f"\n✅ Archivo Excel guardado: {output_path}")
print(f"   → {len(df_unico)} propiedades en {len(html_files)} páginas HTML")

Total de archivos en el ZIP: 108
Archivos HTML encontrados: 108
  Procesados 10/108 archivos... (300 propiedades)
  Procesados 20/108 archivos... (585 propiedades)
  Procesados 30/108 archivos... (885 propiedades)
  Procesados 40/108 archivos... (1185 propiedades)
  Procesados 50/108 archivos... (1485 propiedades)
  Procesados 60/108 archivos... (1785 propiedades)
  Procesados 70/108 archivos... (2085 propiedades)
  Procesados 80/108 archivos... (2385 propiedades)
  Procesados 90/108 archivos... (2685 propiedades)
  Procesados 100/108 archivos... (2985 propiedades)

Total propiedades extraídas (con duplicados): 3201
Propiedades únicas: 3192

Distribución por tipo:
tipo
Departamento    2791
Casa             401
Name: count, dtype: int64

Distribución por ubicación:
ubicacion
Villa Urquiza       1276
Villa Devoto         647
Saavedra             520
Villa del Parque     507
Villa Pueyrredón     242
Name: count, dtype: int64

✅ Archivo Excel guardado: zonaprop_propiedades.xlsx
   → 3192 p

Total de archivos en el ZIP: 108
Archivos HTML encontrados: 108
  Procesados 10/108 archivos... (300 propiedades)
  Procesados 20/108 archivos... (585 propiedades)
  Procesados 30/108 archivos... (885 propiedades)
  Procesados 40/108 archivos... (1185 propiedades)
  Procesados 50/108 archivos... (1485 propiedades)
  Procesados 60/108 archivos... (1785 propiedades)
  Procesados 70/108 archivos... (2085 propiedades)
  Procesados 80/108 archivos... (2385 propiedades)
  Procesados 90/108 archivos... (2685 propiedades)
  Procesados 100/108 archivos... (2985 propiedades)

Total propiedades extraídas (con duplicados): 3201
Propiedades únicas: 3192

Distribución por tipo:
tipo
Departamento    2791
Casa             401
Name: count, dtype: int64

Distribución por ubicación:
ubicacion
Villa Urquiza       1276
Villa Devoto         647
Saavedra             520
Villa del Parque     507
Villa Pueyrredón     242
Name: count, dtype: int64

Propiedades tras filtrar extremos de m²: 2862

── Promedio USD/

/tmp/ipykernel_8443/2407775741.py:162: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(recortar_m2)



✅ Archivo Excel guardado: zonaprop_propiedades.xlsx
   → 3192 propiedades | 181 oportunidades detectadas


In [1]:
"""
Zonaprop Scraper - Extrae propiedades de archivos HTML descargados de Zonaprop
Ejecutar en Google Colab con los archivos HTML en base_zonaprop.zip
"""

import zipfile
import re
import pandas as pd
from bs4 import BeautifulSoup

# ── 1. Leer el ZIP ────────────────────────────────────────────────────────────
ruta_zip = "base_zonaprop.zip"

with zipfile.ZipFile(ruta_zip, 'r') as z:
    archivos = z.namelist()

print(f"Total de archivos en el ZIP: {len(archivos)}")


# ── 2. Funciones auxiliares ───────────────────────────────────────────────────

def detectar_tipo(titulo_html: str) -> str:
    """Detecta si el HTML corresponde a Casas o Departamentos según el título."""
    t = titulo_html.lower()
    if "casas" in t:
        return "Casa"
    if "departamentos" in t or "departamento" in t:
        return "Departamento"
    return "Desconocido"


def limpiar_precio(texto: str) -> int | None:
    """Convierte 'USD 290.000' → 290000. Descarta si no tiene USD."""
    if not texto or "USD" not in texto.upper():
        return None
    nums = re.sub(r"[^\d]", "", texto)
    return int(nums) if nums else None


def extraer_feature(spans, keyword: str) -> int | None:
    """Busca 'N keyword' en la lista de spans y devuelve N como int."""
    for s in spans:
        txt = s.get_text(strip=True)
        if keyword in txt:
            num = re.search(r"(\d+(?:[.,]\d+)?)", txt)
            if num:
                return int(float(num.group(1).replace(",", ".")))
    return None


def parsear_html(nombre_archivo: str, contenido_bytes: bytes) -> list[dict]:
    """Parsea un HTML de Zonaprop y devuelve lista de dicts con las propiedades."""
    soup = BeautifulSoup(contenido_bytes, "html.parser")

    # Tipo: del <title> de la página
    title_tag = soup.find("title")
    tipo = detectar_tipo(title_tag.get_text() if title_tag else "")

    resultados = []

    # Cada propiedad está en un div con data-qa="posting PROPERTY" o "posting DEVELOPMENT"
    cards = soup.find_all("div", attrs={"data-qa": re.compile(r"^posting ")})

    for card in cards:
        datos = {"tipo": tipo, "html": nombre_archivo}

        # ── Precio ────────────────────────────────────────────────────────────
        precio_tag = card.find(attrs={"data-qa": "POSTING_CARD_PRICE"})
        if precio_tag:
            datos["valor"] = limpiar_precio(precio_tag.get_text(strip=True))
        else:
            datos["valor"] = None

        # ── Features (m², ambientes, dorm., baños, cocheras) ─────────────────
        features_tag = card.find(attrs={"data-qa": "POSTING_CARD_FEATURES"})
        spans = features_tag.find_all("span") if features_tag else []

        datos["m2"] = extraer_feature(spans, "m²")
        datos["ambientes"] = extraer_feature(spans, "amb.")
        datos["dormitorios"] = extraer_feature(spans, "dorm.")
        datos["baños"] = extraer_feature(spans, "baño")
        datos["cocheras"] = extraer_feature(spans, "coch.")

        # ── Dirección ─────────────────────────────────────────────────────────
        dir_tag = card.find(
            "h4",
            class_=lambda c: c and "location-address" in c and "location-text" not in c
        )
        datos["direccion"] = dir_tag.get_text(strip=True) if dir_tag else None

        # ── Ubicación ─────────────────────────────────────────────────────────
        loc_tag = card.find(attrs={"data-qa": "POSTING_CARD_LOCATION"})
        if loc_tag:
            texto_loc = loc_tag.get_text(strip=True)
            partes = texto_loc.split(",")
            datos["ubicacion"] = partes[0].strip() if partes else texto_loc
        else:
            datos["ubicacion"] = None

        # Solo agregar si tiene al menos precio o dirección
        if datos["valor"] is not None or datos["direccion"] is not None:
            resultados.append(datos)

    return resultados


# ── 3. Procesar todos los archivos ────────────────────────────────────────────

todas_las_propiedades = []

with zipfile.ZipFile(ruta_zip, 'r') as z:
    html_files = [f for f in z.namelist() if f.lower().endswith(".html")]
    print(f"Archivos HTML encontrados: {len(html_files)}")

    for i, nombre in enumerate(html_files, 1):
        try:
            with z.open(nombre) as f:
                contenido = f.read()
            propiedades = parsear_html(nombre, contenido)
            todas_las_propiedades.extend(propiedades)
            if i % 10 == 0:
                print(f"  Procesados {i}/{len(html_files)} archivos... ({len(todas_las_propiedades)} propiedades)")
        except Exception as e:
            print(f"  ⚠ Error en {nombre}: {e}")

print(f"\nTotal propiedades extraídas (con duplicados): {len(todas_las_propiedades)}")


# ── 4. Crear DataFrame y limpiar ─────────────────────────────────────────────

df = pd.DataFrame(todas_las_propiedades, columns=[
    "tipo", "ubicacion", "valor", "m2", "ambientes",
    "dormitorios", "baños", "cocheras", "direccion", "html"
])

# Eliminar duplicados exactos (misma dirección + precio + HTML)
df_unico = df.drop_duplicates(subset=["direccion", "valor", "html"]).reset_index(drop=True)

print(f"Propiedades únicas: {len(df_unico)}")
print(f"\nDistribución por tipo:")
print(df_unico["tipo"].value_counts())
print(f"\nDistribución por ubicación:")
print(df_unico["ubicacion"].value_counts())


# ── 5. Análisis de valor por metro cuadrado ───────────────────────────────────

# Trabajar solo con filas que tengan valor y m2 válidos y positivos
df_analisis = df_unico.dropna(subset=["valor", "m2"]).copy()
df_analisis = df_analisis[df_analisis["m2"] > 0].reset_index(drop=True)

# Eliminar extremos de m2: recorte por percentil 5% inferior y 95% superior
# aplicado por grupo (tipo + ubicacion) para que la comparación sea justa
def recortar_m2(grupo):
    p5  = grupo["m2"].quantile(0.05)
    p95 = grupo["m2"].quantile(0.95)
    return grupo[(grupo["m2"] >= p5) & (grupo["m2"] <= p95)]

df_analisis = (
    df_analisis
    .groupby(["tipo", "ubicacion"], group_keys=False)
    .apply(recortar_m2)
    .reset_index(drop=True)
)

print(f"\nPropiedades tras filtrar extremos de m²: {len(df_analisis)}")

# Calcular precio por m2 para cada propiedad
df_analisis["usd_m2"] = (df_analisis["valor"] / df_analisis["m2"]).round(0).astype(int)

# Promedio y desvío estándar de usd/m2 por grupo (tipo + ubicacion)
stats = (
    df_analisis
    .groupby(["tipo", "ubicacion"])["usd_m2"]
    .agg(promedio_usd_m2="mean", std_usd_m2="std", n_propiedades="count")
    .reset_index()
)
stats["promedio_usd_m2"] = stats["promedio_usd_m2"].round(0).astype(int)
stats["std_usd_m2"]      = stats["std_usd_m2"].fillna(0).round(0).astype(int)

print("\n── Promedio USD/m² por tipo y ubicación ──")
print(stats.to_string(index=False))

# Unir estadísticas al dataframe de análisis
df_analisis = df_analisis.merge(
    stats[["tipo", "ubicacion", "promedio_usd_m2", "std_usd_m2"]],
    on=["tipo", "ubicacion"], how="left"
)

# Oportunidades: propiedades cuyo usd_m2 está >= UMBRAL_STD desvíos por debajo del promedio
# Con 1.5 desvíos se captura aprox. el 7% inferior (distribución normal)
UMBRAL_STD = 1.5

df_analisis["desvios_bajo_prom"] = (
    (df_analisis["promedio_usd_m2"] - df_analisis["usd_m2"]) / df_analisis["std_usd_m2"]
).round(2)

df_oportunidades = (
    df_analisis[df_analisis["desvios_bajo_prom"] >= UMBRAL_STD]
    .sort_values(["tipo", "ubicacion", "desvios_bajo_prom"], ascending=[True, True, False])
    .reset_index(drop=True)
)

print(f"\n── Oportunidades (≥{UMBRAL_STD} desvíos bajo el promedio): {len(df_oportunidades)} propiedades ──")
cols_print = ["tipo", "ubicacion", "valor", "m2", "usd_m2", "promedio_usd_m2", "desvios_bajo_prom", "direccion"]
print(df_oportunidades[cols_print].to_string(index=False))


# ── 6. Exportar a Excel ───────────────────────────────────────────────────────

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

output_path = "zonaprop_propiedades.xlsx"

wb = Workbook()

# Helpers de estilo
def make_header(ws, columnas, encabezados, color_fondo="2E4057"):
    thin = Side(style="thin", color="CCCCCC")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    hfill = PatternFill("solid", start_color=color_fondo)
    hfont = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    for col_idx, enc in enumerate(encabezados, 1):
        cell = ws.cell(row=1, column=col_idx, value=enc)
        cell.font = hfont
        cell.fill = hfill
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border
    ws.row_dimensions[1].height = 20
    return border

def escribir_filas(ws, df_src, columnas, border, cols_centradas):
    fill_par   = PatternFill("solid", start_color="F5F5F5")
    fill_impar = PatternFill("solid", start_color="FFFFFF")
    font_data  = Font(name="Arial", size=9)
    for row_idx, row in df_src.iterrows():
        fill = fill_par if (row_idx % 2 == 0) else fill_impar
        for col_idx, col in enumerate(columnas, 1):
            val = row[col]
            if not isinstance(val, str) and pd.isna(val):
                val = None
            cell = ws.cell(row=row_idx + 2, column=col_idx, value=val)
            cell.font = font_data
            cell.fill = fill
            cell.border = border
            if col_idx in cols_centradas:
                cell.alignment = Alignment(horizontal="center")


# ── Hoja 1: Todas las propiedades ────────────────────────────────────────────
ws1 = wb.active
ws1.title = "Propiedades"

cols1 = ["tipo", "ubicacion", "valor", "m2", "ambientes", "dormitorios", "baños", "cocheras", "direccion", "html"]
enc1  = ["Tipo", "Ubicación", "Valor (USD)", "m² Totales", "Ambientes", "Dormitorios", "Baños", "Cocheras", "Dirección", "Archivo HTML"]
border1 = make_header(ws1, cols1, enc1)
escribir_filas(ws1, df_unico, cols1, border1, cols_centradas={3,4,5,6,7,8})
for col_idx, ancho in {1:14,2:20,3:14,4:12,5:12,6:14,7:10,8:12,9:35,10:60}.items():
    ws1.column_dimensions[ws1.cell(row=1, column=col_idx).column_letter].width = ancho


# ── Hoja 2: Promedios USD/m² por tipo y ubicación ────────────────────────────
ws_stats = wb.create_sheet("USD_m2_por_zona")

cols_s = ["tipo", "ubicacion", "promedio_usd_m2", "std_usd_m2", "n_propiedades"]
enc_s  = ["Tipo", "Ubicación", "Promedio USD/m²", "Desvío estándar", "N propiedades"]
border_s = make_header(ws_stats, cols_s, enc_s, color_fondo="1B4332")
escribir_filas(ws_stats, stats.reset_index(drop=True), cols_s, border_s, cols_centradas={3,4,5})
for col_idx, ancho in {1:14,2:20,3:18,4:18,5:16}.items():
    ws_stats.column_dimensions[ws_stats.cell(row=1, column=col_idx).column_letter].width = ancho


# ── Hoja 3: Oportunidades ─────────────────────────────────────────────────────
ws_op = wb.create_sheet("Oportunidades")

cols_o = ["tipo", "ubicacion", "valor", "m2", "usd_m2", "promedio_usd_m2",
          "desvios_bajo_prom", "ambientes", "dormitorios", "baños", "cocheras", "direccion", "html"]
enc_o  = ["Tipo", "Ubicación", "Valor (USD)", "m² Totales", "USD/m²", "Promedio USD/m² zona",
          "Desvíos bajo prom.", "Ambientes", "Dormitorios", "Baños", "Cocheras", "Dirección", "Archivo HTML"]
border_o = make_header(ws_op, cols_o, enc_o, color_fondo="7B2D00")

# Colorear filas de oportunidades según qué tan baratas son
fill_muy   = PatternFill("solid", start_color="FFD700")  # dorado  ≥ 2.5 desvíos
fill_buena = PatternFill("solid", start_color="C8F7C5")  # verde   1.5–2.5
thin = Side(style="thin", color="CCCCCC")
border_op  = Border(left=thin, right=thin, top=thin, bottom=thin)
font_data  = Font(name="Arial", size=9)

for row_idx, row in df_oportunidades.iterrows():
    desv = row["desvios_bajo_prom"]
    fill = fill_muy if desv >= 2.5 else fill_buena
    for col_idx, col in enumerate(cols_o, 1):
        val = row[col]
        if not isinstance(val, str) and pd.isna(val):
            val = None
        cell = ws_op.cell(row=row_idx + 2, column=col_idx, value=val)
        cell.font = font_data
        cell.fill = fill
        cell.border = border_op
        if col_idx in {3,4,5,6,7,8,9,10,11}:
            cell.alignment = Alignment(horizontal="center")

for col_idx, ancho in {1:14,2:20,3:14,4:12,5:12,6:22,7:20,8:12,9:14,10:10,11:12,12:35,13:60}.items():
    ws_op.column_dimensions[ws_op.cell(row=1, column=col_idx).column_letter].width = ancho

# Leyenda de colores en oportunidades
ws_op.cell(row=len(df_oportunidades)+4, column=1, value="Leyenda:").font = Font(bold=True, name="Arial", size=9)
c1 = ws_op.cell(row=len(df_oportunidades)+5, column=1, value="≥ 2.5 desvíos bajo el promedio (muy por debajo)")
c1.fill = fill_muy
c1.font = Font(name="Arial", size=9)
c2 = ws_op.cell(row=len(df_oportunidades)+6, column=1, value="1.5–2.5 desvíos bajo el promedio")
c2.fill = fill_buena
c2.font = Font(name="Arial", size=9)


# ── Hoja 4: Resumen ──────────────────────────────────────────────────────────
ws_res = wb.create_sheet("Resumen")
ws_res["A1"] = "Resumen de propiedades extraídas"
ws_res["A1"].font = Font(bold=True, name="Arial", size=12)

filas_res = [
    (3,  "Total propiedades",             len(df_unico)),
    (4,  "Casas",                          int((df_unico["tipo"]=="Casa").sum())),
    (5,  "Departamentos",                  int((df_unico["tipo"]=="Departamento").sum())),
    (7,  "Propiedades analizadas (sin extremos m²)", len(df_analisis)),
    (8,  "Oportunidades detectadas",       len(df_oportunidades)),
    (10, "Precio promedio USD",            f"=AVERAGE(Propiedades!C2:C{len(df_unico)+1})"),
    (11, "Precio mínimo USD",              f"=MIN(Propiedades!C2:C{len(df_unico)+1})"),
    (12, "Precio máximo USD",              f"=MAX(Propiedades!C2:C{len(df_unico)+1})"),
    (14, "m² promedio",                    f"=AVERAGE(Propiedades!D2:D{len(df_unico)+1})"),
    (15, f"Umbral oportunidades (desvíos)", UMBRAL_STD),
]
for fila, etiqueta, valor in filas_res:
    ws_res.cell(row=fila, column=1, value=etiqueta).font = Font(name="Arial", size=10)
    ws_res.cell(row=fila, column=2, value=valor).font   = Font(bold=True, name="Arial", size=10)

ws_res.column_dimensions["A"].width = 40
ws_res.column_dimensions["B"].width = 18

wb.save(output_path)
print(f"\n✅ Archivo Excel guardado: {output_path}")
print(f"   → {len(df_unico)} propiedades | {len(df_oportunidades)} oportunidades detectadas")

Total de archivos en el ZIP: 108
Archivos HTML encontrados: 108
  Procesados 10/108 archivos... (300 propiedades)
  Procesados 20/108 archivos... (585 propiedades)
  Procesados 30/108 archivos... (885 propiedades)
  Procesados 40/108 archivos... (1185 propiedades)
  Procesados 50/108 archivos... (1485 propiedades)
  Procesados 60/108 archivos... (1785 propiedades)
  Procesados 70/108 archivos... (2085 propiedades)
  Procesados 80/108 archivos... (2385 propiedades)
  Procesados 90/108 archivos... (2685 propiedades)
  Procesados 100/108 archivos... (2985 propiedades)

Total propiedades extraídas (con duplicados): 3201
Propiedades únicas: 3192

Distribución por tipo:
tipo
Departamento    2791
Casa             401
Name: count, dtype: int64

Distribución por ubicación:
ubicacion
Villa Urquiza       1276
Villa Devoto         647
Saavedra             520
Villa del Parque     507
Villa Pueyrredón     242
Name: count, dtype: int64

Propiedades tras filtrar extremos de m²: 2862

── Promedio USD/

/tmp/ipykernel_4579/2407775741.py:162: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(recortar_m2)



✅ Archivo Excel guardado: zonaprop_propiedades.xlsx
   → 3192 propiedades | 181 oportunidades detectadas
